In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Deploy the store agent to Agent Runtime and promote a release

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Agent Runtime

[Agent Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) is a managed service that runs your agent for you. You give it the agent object, the Python packages the agent needs, its environment variables and the service account it runs as. Agent Runtime builds a container, starts it and gives you an endpoint that keeps sessions for each user.

Every time you update a deployed agent, Agent Runtime creates a new **revision**. A revision is a fixed copy of the code and configuration. The engine's traffic setting decides which revision answers: always the newest one, or a split between named revisions.

### From a laptop to production

In this workshop you deploy to your own development engine from this notebook. Pre-production and production deploys run from [Cloud Build](https://cloud.google.com/build/docs/overview), with an approval before each step. The code that deploys is the same in both places: the pipeline runs the same calls you run here.

<img width="60%" src="../docs/diagrams/agent-lifecycle.png" alt="The release lifecycle of the store agent, from a pull request to production traffic" />

### Objectives

In this tutorial, you will learn how to deploy an ADK agent to Agent Runtime and how a release moves from development to production.

You will complete the following tasks:

- Package the store agent with its requirements, environment variables and service account
- Start your own MCP server, the only path from the deployed agent to store data
- Create or update your own engine and list its revisions
- Query the deployed agent and list its sessions
- Create an online monitor that scores the agent's live traffic
- Split traffic between two revisions, the way a production canary works
- Read the Cloud Build pipeline that deploys and promotes a release

The notebook takes about ten minutes to run. Most of that is the engine update, seven to eight minutes.

### Costs

This tutorial uses billable components of Google Cloud:

- Agent Runtime on Vertex AI
- Gemini on Vertex AI
- BigQuery
- Cloud Storage

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing), [BigQuery pricing](https://cloud.google.com/bigquery/pricing) and [Cloud Storage pricing](https://cloud.google.com/storage/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This notebook runs against the store data you loaded in the earlier notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The agent's code lives one folder up from this notebook
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

Agent Runtime is a regional service. The store agent runs in `us-central1`, and deployments are uploaded to a staging bucket in your project:

In [3]:
LOCATION = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-cymbal-store-ops-staging"
SERVICE_ACCOUNT = f"store-ops-dev-runtime@{PROJECT_ID}.iam.gserviceaccount.com"

### Import libraries

In [4]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, and the Gen AI SDK logs a note whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

import json
import time

import agentplatform
import google.auth
import yaml
from agentplatform.agent_engines import AdkApp
from google.auth.transport.requests import AuthorizedSession
from google.genai.types import HttpOptions

from agents.cymbal_store_ops.agent import create_app
from agents.cymbal_store_ops.artifact_storage import create_artifact_service

### Create a client

Revisions and traffic splits are part of the `v1beta1` API, so the client pins that version:

In [5]:
client = agentplatform.Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1beta1"),
)

## Prepare the deployment

A deployment has four parts: the agent, the packages it needs, its configuration and the identity it runs as.

### Export the requirements

Agent Runtime installs packages with `pip`. Export the exact versions from the repository's lock file, so the engine runs the same versions you tested with:

In [6]:
os.chdir(REPO_ROOT)  # the package paths below are relative to the repository root

!uv export --quiet --frozen --no-dev --no-emit-project --no-hashes --no-editable --no-header --no-annotate -o build/requirements.txt

requirements = Path("build/requirements.txt").read_text().splitlines()
print(f"{len(requirements)} pinned packages, for example:")
print("\n".join(line for line in requirements if line.startswith(("google-adk", "google-genai"))))

157 pinned packages, for example:
google-adk==2.9.0
google-genai==2.23.0


### Set the configuration

The configuration tells Agent Runtime how to build and run the container:

- `requirements` and `extra_packages`: the packages to install and the local folders to upload. The `agents` folder is the agent's code.
- `env_vars`: the environment the agent reads at startup, such as your namespace, which selects your BigQuery dataset. The `OTEL_*` settings log each prompt and response, which evaluation and monitoring need.
- `service_account`: the identity the agent runs as. It has read access to your dataset and nothing else.
- `labels`: how you find this engine again. The next deploy looks the engine up by these labels and updates it.
- `min_instances`: one instance stays warm, so the first question does not wait for a container to start.
- `traffic_config`: send all traffic to the newest revision. Production uses a manual split instead.

In [7]:
labels = {"app": "cymbal-store-ops", "ns": WORKSHOP_NAMESPACE, "env": "dev"}

config = {
    "display_name": f"cymbal-store-ops-{WORKSHOP_NAMESPACE}-dev",
    "description": "Cymbal Beauty store operations (dev)",
    "requirements": "build/requirements.txt",
    "extra_packages": ["agents", "installation_scripts/install_report_browser.sh"],
    "build_options": {"installation_scripts": ["installation_scripts/install_report_browser.sh"]},
    "staging_bucket": STAGING_BUCKET,
    "gcs_dir_name": f"agent_engine/{WORKSHOP_NAMESPACE}/store-ops-dev",
    "env_vars": {
        "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
        "STORE_OPS_ENV": "dev",
        "WORKSHOP_NAMESPACE": WORKSHOP_NAMESPACE,
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
        # record each prompt and response so traces can be evaluated and monitored
        "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",
        "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "SPAN_AND_EVENT",
        "OTEL_INSTRUMENTATION_GENAI_UPLOAD_FORMAT": "jsonl",
        "OTEL_INSTRUMENTATION_GENAI_COMPLETION_HOOK": "upload",
        "OTEL_INSTRUMENTATION_GENAI_UPLOAD_BASE_PATH": f"gs://{PROJECT_ID}-cymbal-artifacts-{WORKSHOP_NAMESPACE}-dev/genai-payloads",
        "STORE_OPS_PREWARM": "1",
        "CYMBAL_ARTIFACT_BUCKET": f"{PROJECT_ID}-cymbal-artifacts-{WORKSHOP_NAMESPACE}-dev",
        "PLAYWRIGHT_BROWSERS_PATH": "/opt/cymbal-browsers",
    },
    "service_account": SERVICE_ACCOUNT,
    "labels": labels,
    "min_instances": 1,
    "traffic_config": {"traffic_split_always_latest": {}},
}

## Deploy your MCP server

The deployed store agent reads store data only through an [MCP](https://modelcontextprotocol.io/) server: a small Cloud Run service that offers the agent's 30 read tools and checks, on every call, which workload is calling and which store the person is signed in to. Each namespace has its own server, reading its own dataset. The container image is already built, so this section only configures and starts your copy.

<img width="60%" src="../docs/diagrams/store-agent-architecture.png" alt="The store agent reads every store record through the MCP server" />

### Set the MCP service details

The service name and its address follow from your namespace. Cloud Run gives every service a fixed address made of its name, your project number and the region, and the agent uses that address as the audience of the identity token it sends:

In [8]:
MCP_IMAGE = f"us-central1-docker.pkg.dev/{PROJECT_ID}/cymbal-store-ops/mcp-demo-dev:20260923-allreads"
MCP_SERVICE = f"cymbal-store-mcp-{WORKSHOP_NAMESPACE}-dev"
MCP_SECRET = f"cymbal-store-mcp-{WORKSHOP_NAMESPACE}-dev-scope"

project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
MCP_AUDIENCE = f"https://{MCP_SERVICE}-{project_number[0]}.{LOCATION}.run.app"
print(MCP_AUDIENCE)

https://cymbal-store-mcp-opsreview-dev-763419985448.us-central1.run.app


### Create the signing key

The agent signs each request with the person's user, store and role, and the server checks the signature. The key lives in Secret Manager; neither side has it in its configuration. This cell creates the secret and a random key the first time, reuses that key when you run it again, and lets the service account read it:

In [9]:
import secrets

from google.api_core.exceptions import AlreadyExists
from google.cloud import secretmanager

secret_client = secretmanager.SecretManagerServiceClient()
parent = f"projects/{PROJECT_ID}"
try:
    secret_client.create_secret(
        parent=parent,
        secret_id=MCP_SECRET,
        secret={"replication": {"automatic": {}}, "labels": {"app": "cymbal-store-mcp", "ns": WORKSHOP_NAMESPACE}},
    )
except AlreadyExists:
    pass

# Reuse the newest enabled key if this cell has run before; otherwise add the first one
existing = [
    v for v in secret_client.list_secret_versions(parent=f"{parent}/secrets/{MCP_SECRET}")
    if v.state == secretmanager.SecretVersion.State.ENABLED
]
if existing:
    version = existing[0]
else:
    version = secret_client.add_secret_version(
        parent=f"{parent}/secrets/{MCP_SECRET}",
        payload={"data": secrets.token_urlsafe(48).encode()},
    )
MCP_SECRET_VERSION = version.name.rsplit("/", 1)[-1]
print(f"{MCP_SECRET} version {MCP_SECRET_VERSION}")

!gcloud secrets add-iam-policy-binding {MCP_SECRET} --project={PROJECT_ID} --member=serviceAccount:{SERVICE_ACCOUNT} --role=roles/secretmanager.secretAccessor --quiet --format=none

cymbal-store-mcp-opsreview-dev-scope version 3


Updated IAM policy for secret [cymbal-store-mcp-opsreview-dev-scope].


The output names your secret and the key version the server uses. It shows version 3 here because this namespace's secret already had keys; a first run adds version 1.

### Start the service

`gcloud run deploy` starts your copy of the server. The flags that matter:

- `--no-allow-unauthenticated`: only identities with the Cloud Run invoker role can reach it.
- `--add-custom-audiences`: the token audience the agent uses, your service's address.
- `CYMBAL_MCP_TRUSTED_CALLERS`: the only service account the server accepts calls from, your agent's.
- `--set-secrets`: the signing key, pinned to the version you just created.
- `--min-instances=1`: one instance stays warm, so a question never waits for a container to start.

It takes under a minute (34 seconds in the run shown). `gcloud` prints a progress line for every step, so the cell keeps only the last three, which say whether the service is serving:

In [10]:
env = ",".join([
    f"GOOGLE_CLOUD_PROJECT={PROJECT_ID}",
    f"WORKSHOP_NAMESPACE={WORKSHOP_NAMESPACE}",
    "STORE_OPS_ENV=dev",
    f"CYMBAL_MCP_AUDIENCE={MCP_AUDIENCE}",
    f"CYMBAL_MCP_TRUSTED_CALLERS={SERVICE_ACCOUNT}",
    "STORE_OPS_PREWARM=0",
])

# gcloud prints a progress line for every step; keep the last three, which say whether it is serving
!gcloud run deploy {MCP_SERVICE} --project={PROJECT_ID} --region={LOCATION} --image={MCP_IMAGE} --service-account={SERVICE_ACCOUNT} --no-allow-unauthenticated --add-custom-audiences={MCP_AUDIENCE} --set-env-vars={env} --set-secrets=CYMBAL_MCP_SCOPE_KEY={MCP_SECRET}:{MCP_SECRET_VERSION} --min-instances=1 --max-instances=3 --labels=app=cymbal-store-mcp,ns={WORKSHOP_NAMESPACE},env=dev --quiet 2>&1 | tail -3

Service [cymbal-store-mcp-opsreview-dev] revision [cymbal-store-mcp-opsreview-dev-00006-km9] has been deployed and is serving 100 percent of traffic.
Service URL: https://cymbal-store-mcp-opsreview-dev-763419985448.us-central1.run.app
Proxy locally with: gcloud run services proxy cymbal-store-mcp-opsreview-dev --region us-central1 --project mattrobn-sandbox


The deploy ends with "Service [...] has been deployed and is serving 100 percent of traffic" and the service URL, which is the address printed two cells above.

Let the agent's service account call the service:

In [11]:
!gcloud run services add-iam-policy-binding {MCP_SERVICE} --project={PROJECT_ID} --region={LOCATION} --member=serviceAccount:{SERVICE_ACCOUNT} --role=roles/run.invoker --quiet --format=none

Updated IAM policy for service [cymbal-store-mcp-opsreview-dev].


### Point the agent at the server

Four settings switch the agent's reads to the server. The signing key is a reference to the pinned Secret Manager version, never the key itself. With `CYMBAL_MCP_URL` set, the coordinator, each specialist and the opening plan's readers all read through the server:

In [12]:
config["env_vars"].update({
    "CYMBAL_MCP_URL": f"{MCP_AUDIENCE}/mcp",
    "CYMBAL_MCP_AUDIENCE": MCP_AUDIENCE,
    "CYMBAL_MCP_CALLER_SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "CYMBAL_MCP_SCOPE_KEY": {"secret": MCP_SECRET, "version": MCP_SECRET_VERSION},
})

### Build the agent

`create_app()` builds the same app you ran locally in the earlier notebooks. The object you build here is what gets uploaded, so build it now, with the MCP settings in this notebook's environment: the coordinator and each specialist then get their MCP tool connections instead of their local reads. `AdkApp` wraps the app for Agent Runtime and adds the session handling. The artifact service is where the agent saves the PDF reports it creates, in a private Cloud Storage bucket. `enable_tracing=True` keeps the prompt on ADK's `call_llm` spans, which the online monitor in this notebook needs to score answers.

In [13]:
for name in ("CYMBAL_MCP_URL", "CYMBAL_MCP_AUDIENCE", "CYMBAL_MCP_CALLER_SERVICE_ACCOUNT"):
    os.environ[name] = config["env_vars"][name]

app = create_app()
adk_app = AdkApp(app=app, artifact_service_builder=create_artifact_service, enable_tracing=True)

for agent in (app.root_agent, *app.root_agent.sub_agents):
    remote = [tool for tool in agent.tools if "McpToolset" in type(tool).__name__]
    print(f"{agent.name:26} reads through MCP: {bool(remote)}")

store_manager_agent        reads through MCP: True
inventory_excellence       reads through MCP: True
associate_orchestration    reads through MCP: True
loss_prevention            reads through MCP: True
associate_development      reads through MCP: True
store_tasks                reads through MCP: True


All six agents, the coordinator and its five specialists, should print `True`.

## Deploy the agent

### Find your engine

Look for an engine that carries your labels. If one exists, the deploy updates it and creates a new revision. If not, the deploy creates the engine:

In [14]:
existing = [
    engine.api_resource.name
    for engine in client.agent_engines.list()
    if all((engine.api_resource.labels or {}).get(key) == value for key, value in labels.items())
]
engine_name = existing[0] if existing else None
print(engine_name or "No engine yet: the next cell creates one.")

projects/763419985448/locations/us-central1/reasoningEngines/7051567741404184576


### Create or update the engine

This call uploads the code, builds the container and starts it. It takes seven to eight minutes (7 minutes in the run shown). Agent Runtime accepts one update at a time for each engine, so run the cell once and wait for it to finish.

In [15]:
if engine_name:
    remote_agent = client.agent_engines.update(name=engine_name, agent=adk_app, config=config)
else:
    remote_agent = client.agent_engines.create(agent=adk_app, config=config)

engine_name = remote_agent.api_resource.name
print(engine_name)

projects/763419985448/locations/us-central1/reasoningEngines/7051567741404184576


The cell prints the engine's resource name. An update keeps the same name; the number at the end is your engine ID.

### List the revisions

Each update added a revision. The newest one is serving, because the traffic setting is `always latest`:

In [16]:
revisions = sorted(
    client.agent_engines.runtimes.revisions.list(name=engine_name),
    key=lambda revision: revision.api_resource.create_time,
)
for revision in revisions:
    print(revision.api_resource.name.rsplit("/", 1)[-1], revision.api_resource.create_time)

1 2026-09-22 17:17:13.542604+00:00
2 2026-09-22 17:28:01.855805+00:00
3 2026-09-22 18:41:28.084418+00:00
4 2026-09-22 19:09:49.221859+00:00
5 2026-09-22 19:39:17.152296+00:00
6 2026-09-22 20:06:04.634023+00:00
7 2026-09-23 01:52:18.739654+00:00
8 2026-09-23 02:07:17.142723+00:00
9 2026-09-23 04:19:58.824691+00:00
10 2026-09-23 04:31:47.638758+00:00
11 2026-09-23 19:45:41.839559+00:00
12 2026-09-23 22:08:22.913132+00:00
13 2026-09-24 00:56:21.572964+00:00
14 2026-09-24 01:11:35.231560+00:00


The last revision is the one this deploy created. It serves all the traffic. A new engine shows a single revision.

## Query the deployed agent

The deployed agent keeps sessions for you. Create a session as Dana, the manager of store S-014, with the same state the tablet app sets when she signs in:

In [17]:
session = await remote_agent.async_create_session(
    user_id="dana",
    state={
        "user:user_id": "U-M014",
        "user:store_id": "S-014",
        "user:role": "store_manager",
        "user:first_name": "Dana",
    },
)

print(session["id"])

6961020519315406848


Send a question with the [`streamQuery`](https://cloud.google.com/vertex-ai/generative-ai/docs/reference/rest/v1beta1/projects.locations.reasoningEngines/streamQuery) REST method. The response is one JSON event per line: model responses, tool calls, tool results and the final answer.

The Agent Platform SDK also has an `async_stream_query` method. At the SDK version this repository pins, its parser can fail on a tool result that contains braces inside a string, so this notebook reads the stream itself and prints the tool calls and the answer as they arrive:

In [18]:
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
http = AuthorizedSession(credentials)

response = http.post(
    f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{engine_name}:streamQuery",
    json={
        "class_method": "async_stream_query",
        "input": {
            "user_id": "dana",
            "session_id": session["id"],
            "message": "How many units of Lumière Hydra Cream do we have, and where are they?",
        },
    },
    stream=True,
    timeout=600,
)
response.raise_for_status()

for line in response.iter_lines(decode_unicode=True):
    if not line:
        continue
    event = json.loads(line)
    for part in (event.get("content") or {}).get("parts", []):
        if part.get("function_call"):
            print(f"[{event.get('author')}] calls {part['function_call']['name']}")
        elif part.get("text") and not part.get("thought"):
            print(f"\n{part['text']}")

[store_manager_agent] calls list_store_inventory


[store_manager_agent] calls get_inventory_context



We have 7 units of Lumière Hydra Cream (SKU: P-0101) on hand in the store. All 7 units are located in the backroom at Skincare backstock · bay B2, with 0 units on the sales floor shelf (Skincare · fixture SK-04).

Of those 7 units, 4 are actively reserved for 3 pending pickup orders due between 9:30 AM and 10:00 AM, leaving 3 units available to promise.


The question took about 30 seconds. The coordinator, `store_manager_agent`, answered it alone with two reads through your MCP server: `list_store_inventory` and `get_inventory_context`. The answer should give 7 units of Lumière Hydra Cream (P-0101), all in the backroom at Skincare backstock bay B2 and none on the shelf, with 4 held for pickup orders due between 9:30 and 10:00 AM. The store's clock is fixed at Saturday 3 October 2026, 9:00 AM. The wording differs from run to run; the tool calls and the numbers should match.

### List the sessions

Every conversation is stored on the engine. This is where the tablet app's conversations for your namespace live:

In [19]:
for stored in list(client.agent_engines.sessions.list(name=engine_name))[:5]:
    print(stored.name.rsplit("/", 1)[-1], stored.user_id, stored.update_time)

6961020519315406848 dana 2026-09-24 01:12:16.566723+00:00
6703752390601867264 dana 2026-09-24 00:56:59.077017+00:00
7432765983212699648 dana 2026-09-23 22:09:18.033397+00:00
7831897500188409856 dana 2026-09-23 21:55:41.389211+00:00
4371655242740137984 eval-user 2026-09-23 20:53:32.443997+00:00


The first session is the one you just created for Dana. The others are earlier conversations on the same engine, from the tablet app or from evaluation runs (`eval-user`).

## Monitor the agent's live traffic

The gate and the simulated conversations in notebook 04 test the agent before a release. An [online monitor](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online) keeps scoring it afterwards: every ten minutes it samples the engine's recent traces from Cloud Trace and Cloud Logging, scores them with the evaluation service, and writes the scores to Cloud Logging (log `online_evaluator`) and to the agent's evaluation dashboard.

The monitor scores traces that only the coordinator handled. A trace where a specialist or the briefing writer also runs is currently rejected with "system_instruction is not uniform across call_llm spans". This is a platform limit, not a fault in your agent.

A monitor can only score what was logged. The configuration above turned that on: the `OTEL_*` settings record each prompt and response, and upload the full payloads to your artifact bucket. Without them, the console marks traces as "no prompt/response logs" and cannot evaluate them.

### Create a monitor

A monitor names the agent it watches, the share of traces to sample, a cap on how many it scores in one run, and the metrics. It is created with the `onlineEvaluators` REST method:

In [20]:
monitor_request = {
    "displayName": f"Store agent quality ({WORKSHOP_NAMESPACE})",
    "agentResource": engine_name,
    "cloudObservability": {"traceScope": {}, "openTelemetry": {"semconvVersion": "1.39.0"}},
    "config": {"randomSampling": {"percentage": 100}, "maxEvaluatedSamplesPerRun": "20"},
    "metricSources": [
        {"metric": {"predefinedMetricSpec": {"metricSpecName": name}}}
        for name in ("final_response_quality_v1", "tool_use_quality_v1", "hallucination_v1", "safety_v1")
    ],
}

monitors_url = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}/onlineEvaluators"
engine_id = engine_name.rsplit("/", 1)[-1]


def my_monitors() -> list[dict]:
    """The monitors that watch your engine."""
    listed = http.get(monitors_url).json().get("onlineEvaluators", [])
    return [monitor for monitor in listed if monitor["agentResource"].endswith(engine_id)]


# one monitor per engine: running this cell again reuses it
if not my_monitors():
    http.post(monitors_url, json=monitor_request).raise_for_status()
    print("Creating the monitor.")
else:
    print("A monitor already watches this engine; reusing it.")

A monitor already watches this engine; reusing it.


Creating a monitor takes a few seconds. List the monitors on your engine:

In [21]:
time.sleep(15)
monitors = my_monitors()
for monitor in monitors:
    metrics = [source["metric"]["predefinedMetricSpec"]["metricSpecName"] for source in monitor["metricSources"]]
    print(monitor["displayName"], monitor["state"], metrics)

Store agent quality (opsreview) ACTIVE ['final_response_quality_v1', 'tool_use_quality_v1', 'hallucination_v1', 'safety_v1']


The monitor is `ACTIVE` with its four metrics. The cell above prints "Creating the monitor." on the first run. In the run shown the monitor already existed, so it printed "A monitor already watches this engine; reusing it."

Ask your agent a few questions from the tablet app or with the query cell above, then wait ten minutes. Questions the coordinator answers alone, like the stock question above, are the ones the monitor scores.

To see the scores:

- In the Google Cloud console, open **Agent Platform > Agents > Deployments**, select `cymbal-store-ops-<your namespace>-dev` (the display name in the configuration above) and open **Dashboard > Evaluation**. The scores appear as time series, and each scored trace shows its scores on the trace's **Evaluation** tab.
- In **Logging > Logs Explorer**, filter on the log `online_evaluator`. Each scored trace has an entry with its scores; a rejected trace has an entry with the reason, such as "system_instruction is not uniform across call_llm spans".

## Promote a release

In production, a new revision does not take all the traffic at once. The pipeline deploys it with no traffic, sends a small share to it (a canary), watches the results and then moves the rest. A rollback moves the traffic back to a named earlier revision, which takes seconds because nothing is rebuilt.

### Split traffic between two revisions

Send 10 percent of your engine's traffic to the newest revision and 90 percent to the one before it:

In [22]:
newest, previous = revisions[-1].api_resource.name, revisions[-2].api_resource.name

client.agent_engines.update(
    name=engine_name,
    config={
        "traffic_config": {
            "traffic_split_manual": {
                "targets": [
                    {"runtime_revision_name": newest, "percent": 10},
                    {"runtime_revision_name": previous, "percent": 90},
                ]
            }
        }
    },
)

split = client.agent_engines.get(name=engine_name).api_resource.traffic_config.traffic_split_manual
for target in split.targets:
    print(target.runtime_revision_name.rsplit("/", 1)[-1], f"{target.percent}%")

14 10%
13 90%


Revision 14, the newest in this run, now takes 10 percent and revision 13 takes 90 percent. Your revision numbers depend on how many times you have deployed. The update takes about 10 seconds, because no container is rebuilt.

Your development engine should always serve its newest revision, so set the traffic back:

In [23]:
client.agent_engines.update(
    name=engine_name,
    config={"traffic_config": {"traffic_split_always_latest": {}}},
)
print(client.agent_engines.get(name=engine_name).api_resource.traffic_config)

traffic_split_always_latest=ReasoningEngineTrafficConfigTrafficSplitAlwaysLatest() traffic_split_manual=None


`traffic_split_always_latest` is set again and the manual split is gone. New revisions take all the traffic again.

### Follow one change from pull request to production

The cells above are the calls a pipeline makes. The pipeline adds what a laptop cannot: every change is checked the same way, each stage runs under its own service account, and nobody deploys to production by hand. Follow one change, a new sentence in the coordinator's instructions, through the four pipeline files in `cloudbuild/`:

1. **Pull request: `ci.yaml`.** Cloud Build installs the locked dependencies, runs lint and the unit tests, and runs a change review that reads the diff and rates its risk (a high rating fails the check). Then it runs the evaluation gate from notebook 04: the seven gate cases, twice each, against the agent built from the pull request. A prompt change that makes the coverage answer wrong fails here, before anyone merges it.
2. **Merge to main: `deploy.yaml` for dev.** The pipeline writes a release manifest (the commit, the package versions, a digest of the prompts and configuration), deploys a new revision to the dev engine, asks it a real question and runs the evaluation cases against the deployed revision. The manifest is the record of exactly what was tested.
3. **Pre-production: `deploy.yaml` again, after one approval.** The same manifest deploys to the pre-production engine; the deploy refuses a manifest whose commit does not match. The tests run again against that revision.
4. **Production: `deploy.yaml`, then `promote.yaml`.** The new revision is created with no traffic. `promote.yaml` asks it a test question and moves 10 percent of traffic to it, then, after a second approval, 100 percent. `rollback.yaml` moves the traffic back to a named earlier revision in seconds, because nothing is rebuilt.
5. **After the release: the online monitor.** The monitor from this notebook keeps scoring live traffic. It scores the traces the coordinator handled alone; traces where a specialist or the briefing writer also runs are currently rejected, a platform limit. The evaluation cases in `ci.yaml` and `deploy.yaml` cover those paths.

Print the steps of each pipeline to see these stages in the files themselves:

In [24]:
for name in ("ci", "deploy", "promote", "rollback"):
    pipeline = yaml.safe_load(Path(f"cloudbuild/{name}.yaml").read_text())
    steps = [step.get("id") or step["args"][-1].strip().splitlines()[-1][:70] for step in pipeline["steps"]]
    print(f"{name + '.yaml':14} {' -> '.join(steps)}")

ci.yaml        sync -> lint-unit -> change-risk -> eval-gate -> risk-gate
deploy.yaml    release -> deploy -> smoke
promote.yaml   smoke-and-shift-traffic
rollback.yaml  return-traffic


The four files match the stages above: `ci.yaml` ends with the evaluation gate and the risk gate, `deploy.yaml` releases, deploys and tests, and `promote.yaml` and `rollback.yaml` each have one step that moves traffic.

The deploy step runs `deployment/deploy.py`, which makes the calls from the cells above:

In [25]:
pipeline = yaml.safe_load(Path("cloudbuild/deploy.yaml").read_text())

for step in pipeline["steps"]:
    print(f"--- {step['id']}")
    print(step["args"][-1].strip())

--- release
set -euo pipefail
uv sync --locked --all-extras
uv run python deployment/iam/cloudbuild_runtime.py
uv run python deployment/release.py
--- deploy
uv run python deployment/deploy.py --env ${_ENV} --release release.json
--- smoke
set -euo pipefail
REV=$$(uv run python -c "import json; print(json.load(open('deployment/deployment_info.${_NAMESPACE}.${_ENV}.json'))['revision'])")
test -n "$$REV" || { echo "deploy recorded no revision"; exit 1; }
uv run python deployment/smoke.py --env ${_ENV} --revision "$$REV"
uv run python deployment/remote_eval.py --env ${_ENV} --revision "$$REV"


### Approvals

Each stage runs from its own Cloud Build trigger, under its own service account. A trigger with approval required waits until a named person approves the build in the console, or with `gcloud builds approve BUILD_ID`. The trigger definition for the pre-production deploy looks like this:

```json
{
  "name": "NAMESPACE-deploy-preprod",
  "serviceAccount": "projects/PROJECT/serviceAccounts/cicd-deployer-preprod@PROJECT.iam.gserviceaccount.com",
  "gitFileSource": {"path": "cloudbuild/deploy.yaml"},
  "substitutions": {"_NAMESPACE": "NAMESPACE", "_ENV": "preprod"},
  "approvalConfig": {"approvalRequired": true}
}
```

The same commit climbs every stage; only the configuration changes:

| Stage | What runs | Approval |
|---|---|---|
| Pull request | Lint, unit tests, a review of the change, the evaluation cases | None: the checks pass or fail |
| Dev | Deploy on merge, then a test question and the evaluation cases | None |
| Pre-production | The same release, deployed and tested again | One person |
| Production | A new revision with no traffic, then 10 percent, then 100 percent | One approval for each step |

Learn more about [approving builds](https://cloud.google.com/build/docs/securing-builds/gate-builds-on-approval).

## Cleaning up

Keep everything if you go on to the next notebooks. When you have finished with the workshop, delete what this notebook created, in this order, so nothing keeps running or billing.

Delete your monitor, so it stops scoring:

```python
for monitor in monitors:
    http.delete(f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{monitor['name']}")
```

Delete the engine. The `force=True` flag also deletes its sessions:

```python
client.agent_engines.delete(name=engine_name, force=True)
```

Delete your MCP server and its signing key:

```python
!gcloud run services delete {MCP_SERVICE} --project={PROJECT_ID} --region={LOCATION} --quiet
secret_client.delete_secret(name=f"projects/{PROJECT_ID}/secrets/{MCP_SECRET}")
```

## What's next

- [Deploy an agent to Agent Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/deploy)
- [Manage deployed agents](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/manage/overview)
- [Promotion strategy](../docs/PROMOTION_STRATEGY.md) for this repository
- Next notebook: [Governance controls for the store agent](06_governance.ipynb)